In [ ]:
# ================================
# 📦 SILENT SETUP
# ================================
import sys, subprocess, warnings, time, logging
warnings.filterwarnings("ignore")
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

def install_quiet(pkg):
    subprocess.call([sys.executable, "-m", "pip", "install", pkg],
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.DEVNULL)

install_quiet("yfinance")
install_quiet("pandas")
install_quiet("numpy")
install_quiet("pytz")

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import pytz

# ================================
# 🕒 MARKET STATUS
# ================================
def is_market_open():
    now = datetime.now(pytz.timezone('US/Eastern'))
    return 9 <= now.hour < 16

MARKET_OPEN = is_market_open()

# ================================
# 📋 FULL STOCK LIST
# ================================
RAW_LIST = """
A AA AAL AAP AAPL ABBV ABNB ABT ACN ADBE ADI ADM ADP ADSK AEP AES AFL AFRM AGNC AI AIG AKAM ALB ALK ALL
AMAT AMD AMGN AMRN AMZN APA APH APTV AVGO AXP BA BABA BAC BAX BBY BEN BIDU BIIB BITO BK BKR BMY BP BSX BX
BYND C CAH CAT CB CCI CCJ CCL CF CFG CHWY CI CL CLF CLX CMCSA CME CNC CNP COF COIN COP COST CPB CPRI CRM
CRON CRWD CSCO CSX CTVA CVNA CVS CVX CZR D DAL DD DE DELL DHI DHR DIA DIS DKNG DLR DOCU DOW DRI DVN DXC EA
EBAY ED EEM EFA EIX EL EMR ENPH EOG EQR EQT ETSY EVRG EW EWJ EWW EWY EWZ EXC EXPE F FANG FAST FCX FDX FE FI
FITB FOXA FSLR FTI FTV FXE FXI GD GDX GE GILD GLD GLW GM GME GOOG GOOGL GPRO GPS GS HAL HBAN HBI HCA HD HIG
HLT HOG HOLX HON HPE HPQ HYG IBB IBIT IBM ICE IEF INTC IP IPG IRM IVZ IWM IYR JCI JD JETS JNJ JNPR JPM K KHC
KMI KO KR KRE KSS KWEB LEN LI LKQ LLY LNC LOW LQD LUMN LUV LVS LYB LYFT MA MAR MARA MAS MCD MCHP MDLZ MDT MET
META MGM MMM MO MOS MPC MRK MRNA MRVL MS MSFT MSTR MTB MTCH MU NCLH NEE NEM NET NFLX NKE NOW NRG NTAP NTES NVAX
NVDA NWL NWSA O OIH OKE OMC ON ORCL OXY PARA PBR PDD PENN PEP PFE PFG PG PGR PINS PLTR PNC PPL PRU PSX PTON
PYPL QCOM QQQ RBLX RCL RF RIG RIOT RIVN ROKU ROST RTX RUN SBUX SCHD SCHW SEDG SHOP SIRI SLB SLV SMCI SMH SNAP
SNOW SO SOFI SOXL SOXS SPG SPX SPXL SPXS SPY SQQQ STX SWKS SYF SYY T TAP TBT TCOM TDOC TFC TGT TJX TLT TMO TMUS
TPR TQQQ TRIP TRV TSLA TSM TSN TT TTD TTWO TXN U UA UAA UAL UBER ULTA UNG UNH UNP UPS UPST URBN USB USO UVXY V
VALE VFC VGK VTR VXX VZ WAB WBA WBD WDC WFC WM WMB WMT WU WY WYNN X XBI XEL XHB XLB XLC XLE XLF XLI XLK XLP XLRE
XLU XLV XLY XOM XOP XRT XRX XSP YELP YINN ZION ZM
"""
symbols = [s.strip() for s in RAW_LIST.split() if s.strip()]

# ================================
# ⚙️ SETTINGS
# ================================
MAX_STOCKS = 20
BOTTOM_PERCENT = 0.05

stage_counts = {k:0 for k in [
    "total","data_ok","bottom_5pct","no_breakdown","final"
]}

# ================================
# 🔁 FETCH
# ================================
def fetch(symbol, interval, period):
    try:
        df = yf.download(symbol, period=period, interval=interval,
                         progress=False, auto_adjust=True)
        if df is not None and not df.empty:
            return df
    except:
        pass

    try:
        df = pd.read_csv(f"https://stooq.com/q/d/l/?s={symbol.lower()}.us&i=d")
        df['Date'] = pd.to_datetime(df['Date'])
        df.set_index('Date', inplace=True)
        return df
    except:
        return None

# ================================
# 🔍 ANALYZE
# ================================
def analyze(symbol):
    stage_counts["total"] += 1

    df = fetch(symbol, "5m", "5d")
    daily = fetch(symbol, "1d", "10d")

    if df is None or daily is None or df.empty or daily.empty or len(daily) < 2:
        return None
    stage_counts["data_ok"] += 1

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    if isinstance(daily.columns, pd.MultiIndex):
        daily.columns = daily.columns.get_level_values(0)

    # Correct previous day logic
    if MARKET_OPEN:
        prev = daily.iloc[-2]
    else:
        prev = daily.iloc[-1]

    prev_low = float(prev['Low'])
    prev_high = float(prev['High'])

    current = float(df['Close'].iloc[-1])
    today_open = float(df['Open'].iloc[0])

    # 1. Bottom 5%
    position = (current - prev_low) / (prev_high - prev_low + 1e-9)
    if position > BOTTOM_PERCENT:
        return None
    stage_counts["bottom_5pct"] += 1

    # 2. No breakdown
    if today_open < prev_low:
        return None
    stage_counts["no_breakdown"] += 1

    stage_counts["final"] += 1
    return symbol

# ================================
# 🚀 SCAN
# ================================
results = []

for i, s in enumerate(symbols):
    print(f"\rScanning {i+1}/{len(symbols)}...", end="")
    res = analyze(s)
    if res:
        results.append(res)
    time.sleep(0.1)

# ================================
# 📊 OUTPUT
# ================================
print("\n\n📊 STAGE BREAKDOWN:")
for k,v in stage_counts.items():
    print(f"{k}: {v}")

print("\n📊 TRADINGVIEW WATCHLIST:\n")
top = results[:MAX_STOCKS]
print(",".join(top[:10]))
print(",".join(top[10:20]))